In [1]:
import polars as pl

# Load lazily to reduce RAM usage
df = pl.read_parquet("../data/malaysia_transactions.parquet", use_pyarrow=True)

In [2]:
print("Shape:", df.shape)
print("Columns:", df.columns)

Shape: (11998251, 19)
Columns: ['trxn_id', 'date_time', 'ofi_acct_number', 'ofi_bank_code', 'ofi_entity_id', 'ofi_entity_type', 'rfi_acct_number', 'rfi_bank_code', 'rfi_entity_id', 'rfi_entity_type', 'trxn_amount', 'trxn_channel', 'trxn_type', 'service_code', 'service_name', 'ofi_entity_name', 'rfi_entity_name', 'ofi_participant_type', 'rfi_participant_type']


### null count

In [3]:
print("🧯 Null count per column:")
display(df.null_count())

🧯 Null count per column:


trxn_id,date_time,ofi_acct_number,ofi_bank_code,ofi_entity_id,ofi_entity_type,rfi_acct_number,rfi_bank_code,rfi_entity_id,rfi_entity_type,trxn_amount,trxn_channel,trxn_type,service_code,service_name,ofi_entity_name,rfi_entity_name,ofi_participant_type,rfi_participant_type
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,438281,665434,442985,442985,464000,665434,0,0,665434,0,11364827,0,442985,0,0


### Is date_time sorted?

In [4]:
df["date_time"].is_sorted()

False

### Check if same ofi_entity_name → multiple ofi_acct_number

In [14]:
df.select(["ofi_entity_name", "ofi_acct_number"]) \
  .unique() \
  .group_by("ofi_entity_name") \
  .agg(pl.len().alias("num_accounts")) \
  .sort("num_accounts", descending=True) \
  .head(10) 

ofi_entity_name,num_accounts
str,u32
"""Unknown Entity""",398613
"""Ong Ahmad""",5456
"""Nurul binti Omar""",5451
"""Chong Wong""",5446
"""Li Hua Pillai""",5439
"""Priya Ahmad""",5431
"""Yusuf Menon""",5423
"""Ismail Ng""",5421
"""Ismail Ong""",5414


### Check if same ofi_acct_number → multiple ofi_entity_name

In [17]:
df.select(["ofi_entity_name", "ofi_acct_number"]) \
  .unique() \
  .group_by("ofi_acct_number") \
  .agg(pl.len().alias("num_names")) \
  .sort("num_names", descending=True) \
  .head(10)

ofi_acct_number,num_names
str,u32
"""SB172599859656""",2
"""SB017321485862""",2
"""SB012773041259""",2
"""BP860206152897""",2
"""SB179726784038""",2
"""BJ662945828936""",1
"""SB547181407516""",1
"""SB170417121032""",1
"""SB344749055926""",1


### Check if same rfi_entity_name → multiple rfi_acct_number

In [20]:
df.select(["rfi_entity_name", "rfi_acct_number"]) \
  .unique() \
  .group_by("rfi_entity_name") \
  .agg(pl.len().alias("num_accounts")) \
  .sort("num_accounts", descending=True) \
  .head(10)

rfi_entity_name,num_accounts
str,u32
"""Unknown Entity""",15810
"""Siti Hassan""",2673
"""Muhammad Yusof""",2649
"""Khadijah Singh""",2641
"""Ahmad Ibrahim""",2627
"""Siti Devi""",2621
"""Wei Ming Ahmad""",2619
"""Wei Ming Nair""",2614
"""Ali Ibrahim""",2612


### Check if same rfi_acct_number → multiple rfi_entity_name

In [23]:
df.select(["rfi_entity_name", "rfi_acct_number"]) \
  .unique() \
  .group_by("rfi_acct_number") \
  .agg(pl.len().alias("num_names")) \
  .sort("num_names", descending=True) \
  .head(1000)

rfi_acct_number,num_names
str,u32
"""STB126866372697""",452738
"""BP177380680284""",125473
"""PF032790211749""",124812
"""PF493628200973""",90792
"""SB339885003146""",86680
…,…
"""PIB755863608783""",3
"""BP900682098253""",3
"""BP275910383529""",3


In [28]:
ofi_accounts = df.select("ofi_acct_number").unique()
rfi_accounts = df.select("rfi_acct_number").unique()

# Convert to sets for intersection
ofi_set = set(ofi_accounts["ofi_acct_number"].to_list())
rfi_set = set(rfi_accounts["rfi_acct_number"].to_list())

intersection = ofi_set & rfi_set

print("Intersecting accounts:", len(intersection))
print("Total sender (ofi) accounts:", len(ofi_set))
print("Total receiver (rfi) accounts:", len(rfi_set))

Intersecting accounts: 615021
Total sender (ofi) accounts: 7469050
Total receiver (rfi) accounts: 2875835


In [ ]:
edges = df.select([
    pl.col("ofi_acct_number").alias("src"),
    pl.col("rfi_acct_number").alias("dst"),
    pl.col("date_time")  # optional, but keep for now
])

edges_sorted = edges.sort("date_time")

In [1]:
# TODO: EDA: Is the channel and type useful? If no remove in cleaning stage.
# TODO: Cleaning: Remove columns like name and service code/name
# TODO: Create graph, determine existence of cycles, etc.